In [1]:
# CELL 1: Import all required libraries and suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Memory optimization
import gc

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Models
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Evaluation metrics
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")
print(f"📦 Pandas version: {pd.__version__}")
print(f"📦 NumPy version: {np.__version__}")
print(f"📦 XGBoost version: {xgb.__version__}")

✅ All libraries imported successfully!
📦 Pandas version: 2.3.2
📦 NumPy version: 2.3.3
📦 XGBoost version: 3.0.5


In [2]:
# CELL 2: Load dataset with memory optimization
print("📂 Loading NF-UNSW-NB15-v3.csv dataset...")
print("=" * 60)

# Load dataset
df = pd.read_csv('data/NF-UNSW-NB15-v3.csv')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n" + "=" * 60)

# Display basic information
print("\n📋 First 5 rows:")
print(df.head())

print("\n" + "=" * 60)
print("\n🔍 Dataset Info:")
print(df.info())

print("\n" + "=" * 60)
print("\n📊 Column names:")
print(df.columns.tolist())

print("\n" + "=" * 60)
print("\n🎯 Target variable distributions:")
if 'Label' in df.columns:
    print("\n🔹 Binary Label distribution:")
    print(df['Label'].value_counts())
    print(f"   Percentage: \n{df['Label'].value_counts(normalize=True) * 100}")

if 'Attack' in df.columns:
    print("\n🔹 Multi-class Attack distribution:")
    print(df['Attack'].value_counts())
    print(f"   Percentage: \n{df['Attack'].value_counts(normalize=True) * 100}")

# Check for duplicates
print("\n" + "=" * 60)
print(f"\n🔄 Duplicate rows: {df.duplicated().sum()}")

# Force garbage collection
gc.collect()

📂 Loading NF-UNSW-NB15-v3.csv dataset...
✅ Dataset loaded successfully!
📊 Shape: 2,365,424 rows × 55 columns
💾 Memory usage: 1336.16 MB


📋 First 5 rows:
   FLOW_START_MILLISECONDS  FLOW_END_MILLISECONDS IPV4_SRC_ADDR  L4_SRC_PORT  \
0            1424242193040          1424242193043    59.166.0.2         4894   
1            1424242192744          1424242193079    59.166.0.4        52671   
2            1424242190649          1424242193109    59.166.0.0        47290   
3            1424242193145          1424242193146    59.166.0.8        43310   
4            1424242193239          1424242193241    59.166.0.1        45870   

   IPV4_DST_ADDR  L4_DST_PORT  PROTOCOL  L7_PROTO  IN_BYTES  IN_PKTS  ...  \
0  149.171.126.3           53        17       5.0       146        2  ...   
1  149.171.126.6        31992         6      11.0      4704       28  ...   
2  149.171.126.9         6881         6      37.0     13662      238  ...   
3  149.171.126.7           53        17       5.0       1

20

In [3]:
# CELL 3: Check for missing values and infinite values
print("🔍 Checking data quality issues...")
print("=" * 60)

# Check missing values
print("\n📉 Missing values per column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
    print(f"\n⚠️  Total columns with missing values: {len(missing_df)}")
else:
    print("✅ No missing values found!")

# Check for infinite values in numeric columns
print("\n" + "=" * 60)
print("\n♾️  Checking for infinite values in numeric columns...")

numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = {}

for col in numeric_cols:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        inf_counts[col] = inf_count

if inf_counts:
    print(f"\n⚠️  Columns with infinite values:")
    for col, count in sorted(inf_counts.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(df)) * 100
        print(f"   {col}: {count:,} ({pct:.2f}%)")
else:
    print("✅ No infinite values found!")

# Check data types
print("\n" + "=" * 60)
print("\n📊 Data types summary:")
print(df.dtypes.value_counts())

# Check for negative values in throughput columns (might indicate issues)
print("\n" + "=" * 60)
print("\n🔎 Checking for suspicious values in float columns:")
float_cols = df.select_dtypes(include=[np.float64]).columns
for col in float_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"   {col}: {neg_count:,} negative values")
    print(f"   {col} - Min: {df[col].min()}, Max: {df[col].max()}, NaN: {df[col].isna().sum()}")

print("\n" + "=" * 60)
print(f"\n💾 Current memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

🔍 Checking data quality issues...

📉 Missing values per column:
                         Missing_Count  Missing_Percentage
SRC_TO_DST_SECOND_BYTES          63425            2.681337

⚠️  Total columns with missing values: 1


♾️  Checking for infinite values in numeric columns...

⚠️  Columns with infinite values:
   DST_TO_SRC_SECOND_BYTES: 122,493 (5.18%)
   SRC_TO_DST_SECOND_BYTES: 59,068 (2.50%)


📊 Data types summary:
int64      49
object      3
float64     3
Name: count, dtype: int64


🔎 Checking for suspicious values in float columns:
   L7_PROTO - Min: 0.0, Max: 421.0, NaN: 0
   SRC_TO_DST_SECOND_BYTES - Min: 0.0, Max: inf, NaN: 63425
   DST_TO_SRC_SECOND_BYTES - Min: 0.0014166535495041, Max: inf, NaN: 0


💾 Current memory usage: 1336.16 MB


In [4]:
# CELL 4: Clean the dataset - handle duplicates, missing values, and infinite values
print("🧹 Cleaning the dataset...")
print("=" * 60)

# Store original shape
original_shape = df.shape
print(f"📊 Original shape: {original_shape[0]:,} rows × {original_shape[1]} columns")

# Step 1: Remove duplicates
print("\n🔄 Step 1: Removing duplicate rows...")
df_clean = df.drop_duplicates()
print(f"   Removed {original_shape[0] - df_clean.shape[0]:,} duplicate rows")
print(f"   New shape: {df_clean.shape[0]:,} rows")

# Step 2: Handle infinite values - replace with NaN first
print("\n♾️  Step 2: Replacing infinite values with NaN...")
inf_cols = ['SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES']
for col in inf_cols:
    inf_count = np.isinf(df_clean[col]).sum()
    if inf_count > 0:
        df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
        print(f"   {col}: {inf_count:,} infinite values replaced with NaN")

# Step 3: Handle missing values - fill with median for numerical stability
print("\n📉 Step 3: Handling missing values...")
missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
print(f"   Columns with missing values: {missing_cols}")

for col in missing_cols:
    missing_count = df_clean[col].isnull().sum()
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"   {col}: {missing_count:,} missing values filled with median ({median_val:.2f})")

# Step 4: Verify cleaning
print("\n✅ Step 4: Verifying data cleaning...")
print(f"   Missing values remaining: {df_clean.isnull().sum().sum()}")
print(f"   Infinite values remaining: {np.isinf(df_clean.select_dtypes(include=[np.number])).sum().sum()}")
print(f"   Duplicate rows remaining: {df_clean.duplicated().sum()}")

# Final shape
print("\n" + "=" * 60)
print(f"📊 Final cleaned shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"📉 Rows removed: {original_shape[0] - df_clean.shape[0]:,} ({(original_shape[0] - df_clean.shape[0])/original_shape[0]*100:.2f}%)")

# Check target distributions after cleaning
print("\n" + "=" * 60)
print("\n🎯 Target distributions after cleaning:")
print(f"\n🔹 Binary Label:")
print(df_clean['Label'].value_counts())
print(f"\n🔹 Multi-class Attack (top 5):")
print(df_clean['Attack'].value_counts().head())

# Memory cleanup
del df
gc.collect()

print("\n" + "=" * 60)
print(f"💾 Memory usage after cleaning: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("✅ Data cleaning completed!")

🧹 Cleaning the dataset...
📊 Original shape: 2,365,424 rows × 55 columns

🔄 Step 1: Removing duplicate rows...
   Removed 14,815 duplicate rows
   New shape: 2,350,609 rows

♾️  Step 2: Replacing infinite values with NaN...
   SRC_TO_DST_SECOND_BYTES: 59,024 infinite values replaced with NaN
   DST_TO_SRC_SECOND_BYTES: 109,125 infinite values replaced with NaN

📉 Step 3: Handling missing values...
   Columns with missing values: ['SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES']
   SRC_TO_DST_SECOND_BYTES: 109,125 missing values filled with median (162.00)
   DST_TO_SRC_SECOND_BYTES: 109,125 missing values filled with median (75.30)

✅ Step 4: Verifying data cleaning...
   Missing values remaining: 0
   Infinite values remaining: 0
   Duplicate rows remaining: 0

📊 Final cleaned shape: 2,350,609 rows × 55 columns
📉 Rows removed: 14,815 (0.63%)


🎯 Target distributions after cleaning:

🔹 Binary Label:
Label
0    2222930
1     127679
Name: count, dtype: int64

🔹 Multi-class Attack (to

In [7]:
# CELL 5: Encode categorical features and prepare feature/target separation
print("🔧 Encoding categorical features...")
print("=" * 60)

# Identify categorical columns (excluding target variables)
categorical_cols = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']
print(f"\n📝 Categorical columns to encode: {categorical_cols}")

# Create a copy for processing
df_encoded = df_clean.copy()

# Encode IP addresses using Label Encoder
print("\n🔤 Encoding IP addresses...")
le_src = LabelEncoder()
le_dst = LabelEncoder()

df_encoded['IPV4_SRC_ADDR_ENCODED'] = le_src.fit_transform(df_encoded['IPV4_SRC_ADDR'])
df_encoded['IPV4_DST_ADDR_ENCODED'] = le_dst.fit_transform(df_encoded['IPV4_DST_ADDR'])

print(f"   IPV4_SRC_ADDR: {df_encoded['IPV4_SRC_ADDR'].nunique():,} unique IPs encoded")
print(f"   IPV4_DST_ADDR: {df_encoded['IPV4_DST_ADDR'].nunique():,} unique IPs encoded")

# Drop original IP address columns
df_encoded = df_encoded.drop(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR'], axis=1)
print(f"\n Original IP columns dropped")

# Encode Attack labels for multi-class classification
print("\n🎯 Encoding Attack labels...")
le_attack = LabelEncoder()
df_encoded['Attack_Encoded'] = le_attack.fit_transform(df_encoded['Attack'])

print(f"   Attack classes: {le_attack.classes_}")
print(f"   Number of classes: {len(le_attack.classes_)}")

# Show mapping
print("\n   Attack Label Mapping:")
for idx, attack_type in enumerate(le_attack.classes_):
    count = (df_encoded['Attack_Encoded'] == idx).sum()
    print(f"      {idx}: {attack_type} ({count:,} samples)")

# Drop original Attack column
df_encoded = df_encoded.drop(['Attack'], axis=1)

# Separate features and targets
print("\n" + "=" * 60)
print("\n📊 Preparing features and targets...")

# For Binary Classification (Label: 0=Benign, 1=Attack)
y_binary = df_encoded['Label'].values
X = df_encoded.drop(['Label', 'Attack_Encoded'], axis=1).values
feature_names = df_encoded.drop(['Label', 'Attack_Encoded'], axis=1).columns.tolist()

# For Multi-class Classification (Attack_Encoded)
y_multiclass = df_encoded['Attack_Encoded'].values

print(f"\n Feature matrix (X): {X.shape[0]:,} samples × {X.shape[1]} features")
print(f" Binary target (y_binary): {y_binary.shape[0]:,} samples")
print(f"   - Class 0 (Benign): {(y_binary == 0).sum():,} ({(y_binary == 0).sum()/len(y_binary)*100:.2f}%)")
print(f"   - Class 1 (Attack): {(y_binary == 1).sum():,} ({(y_binary == 1).sum()/len(y_binary)*100:.2f}%)")

print(f"\n Multi-class target (y_multiclass): {y_multiclass.shape[0]:,} samples")
print(f"   - Number of classes: {len(np.unique(y_multiclass))}")

print(f"\n📝 Feature names ({len(feature_names)} total):")
print(f"   {feature_names[:10]}...")
print(f"   ... (showing first 10)")

# Memory cleanup
del df_clean
gc.collect()

print("\n" + "=" * 60)
print(f"💾 Memory usage: {df_encoded.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(" Feature encoding completed!")

🔧 Encoding categorical features...

📝 Categorical columns to encode: ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']

🔤 Encoding IP addresses...
   IPV4_SRC_ADDR: 40 unique IPs encoded
   IPV4_DST_ADDR: 40 unique IPs encoded

 Original IP columns dropped

🎯 Encoding Attack labels...
   Attack classes: ['Analysis' 'Backdoor' 'Benign' 'DoS' 'Exploits' 'Fuzzers' 'Generic'
 'Reconnaissance' 'Shellcode' 'Worms']
   Number of classes: 10

   Attack Label Mapping:
      0: Analysis (1,226 samples)
      1: Backdoor (4,658 samples)
      2: Benign (2,222,930 samples)
      3: DoS (5,971 samples)
      4: Exploits (42,744 samples)
      5: Fuzzers (33,816 samples)
      6: Generic (19,651 samples)
      7: Reconnaissance (17,074 samples)
      8: Shellcode (2,381 samples)
      9: Worms (158 samples)


📊 Preparing features and targets...

 Feature matrix (X): 2,350,609 samples × 53 features
 Binary target (y_binary): 2,350,609 samples
   - Class 0 (Benign): 2,222,930 (94.57%)
   - Class 1 (Attack): 127,679 

In [9]:
# CELL 6: Split data into train/test sets for both binary and multi-class tasks
print("✂️  Splitting data into train/test sets...")
print("=" * 60)

# Set random seed for reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2

# BINARY CLASSIFICATION SPLIT
print("\n🔹 Binary Classification (Label: Benign vs Attack)")
print("-" * 60)

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X, y_binary, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=y_binary  # Maintain class distribution
)

print(f"✅ Training set: {X_train_bin.shape[0]:,} samples ({X_train_bin.shape[0]/len(X)*100:.1f}%)")
print(f"   - Benign: {(y_train_bin == 0).sum():,} ({(y_train_bin == 0).sum()/len(y_train_bin)*100:.2f}%)")
print(f"   - Attack: {(y_train_bin == 1).sum():,} ({(y_train_bin == 1).sum()/len(y_train_bin)*100:.2f}%)")

print(f"\n✅ Test set: {X_test_bin.shape[0]:,} samples ({X_test_bin.shape[0]/len(X)*100:.1f}%)")
print(f"   - Benign: {(y_test_bin == 0).sum():,} ({(y_test_bin == 0).sum()/len(y_test_bin)*100:.2f}%)")
print(f"   - Attack: {(y_test_bin == 1).sum():,} ({(y_test_bin == 1).sum()/len(y_test_bin)*100:.2f}%)")

# MULTI-CLASS CLASSIFICATION SPLIT
print("\n" + "=" * 60)
print("\n🔹 Multi-class Classification (Attack Types)")
print("-" * 60)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y_multiclass,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_multiclass  # Maintain class distribution
)

print(f"✅ Training set: {X_train_multi.shape[0]:,} samples ({X_train_multi.shape[0]/len(X)*100:.1f}%)")
print(f"\n   Class distribution in training set:")
for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    count = (y_train_multi == class_idx).sum()
    pct = count / len(y_train_multi) * 100
    print(f"      {class_idx}: {class_name:15s} - {count:,} ({pct:.2f}%)")

print(f"\n✅ Test set: {X_test_multi.shape[0]:,} samples ({X_test_multi.shape[0]/len(X)*100:.1f}%)")
print(f"\n   Class distribution in test set:")
for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    count = (y_test_multi == class_idx).sum()
    pct = count / len(y_test_multi) * 100
    print(f"      {class_idx}: {class_name:15s} - {count:,} ({pct:.2f}%)")

# Verify shapes match
print("\n" + "=" * 60)
print("\n🔍 Verification:")
print(f"   Binary train/test shapes match: {X_train_bin.shape == X_train_multi.shape}")
print(f"   Feature dimension: {X_train_bin.shape[1]} features")

print("\n⚠️  IMBALANCE WARNING:")
print(f"   Binary: 94.57% benign - models may achieve high accuracy by predicting only benign!")
print(f"   Multi-class: Worms class has only ~126 training samples - may be difficult to learn!")

# Memory cleanup
del df_encoded
gc.collect()

print("\n" + "=" * 60)
print("✅ Train/test split completed successfully!")

✂️  Splitting data into train/test sets...

🔹 Binary Classification (Label: Benign vs Attack)
------------------------------------------------------------
✅ Training set: 1,880,487 samples (80.0%)
   - Benign: 1,778,344 (94.57%)
   - Attack: 102,143 (5.43%)

✅ Test set: 470,122 samples (20.0%)
   - Benign: 444,586 (94.57%)
   - Attack: 25,536 (5.43%)


🔹 Multi-class Classification (Attack Types)
------------------------------------------------------------
✅ Training set: 1,880,487 samples (80.0%)

   Class distribution in training set:
      0: Analysis        - 981 (0.05%)
      1: Backdoor        - 3,726 (0.20%)
      2: Benign          - 1,778,344 (94.57%)
      3: DoS             - 4,777 (0.25%)
      4: Exploits        - 34,195 (1.82%)
      5: Fuzzers         - 27,053 (1.44%)
      6: Generic         - 15,721 (0.84%)
      7: Reconnaissance  - 13,659 (0.73%)
      8: Shellcode       - 1,905 (0.10%)
      9: Worms           - 126 (0.01%)

✅ Test set: 470,122 samples (20.0%)

   Cl

In [10]:
# CELL 7: Scale features using StandardScaler
print("⚖️  Scaling features with StandardScaler...")
print("=" * 60)

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both train and test
print("\n🔹 Scaling Binary Classification datasets...")
print("-" * 60)

# Fit scaler on training data only (prevent data leakage)
X_train_bin_scaled = scaler.fit_transform(X_train_bin)
X_test_bin_scaled = scaler.transform(X_test_bin)

print(f"✅ Training set scaled: {X_train_bin_scaled.shape}")
print(f"✅ Test set scaled: {X_test_bin_scaled.shape}")

# Show scaling statistics for first 5 features
print(f"\n📊 Scaling statistics (first 5 features):")
print(f"   Feature means (after scaling): {X_train_bin_scaled[:, :5].mean(axis=0)}")
print(f"   Feature stds (after scaling): {X_train_bin_scaled[:, :5].std(axis=0)}")

# For multi-class, use the same scaler (same features)
print("\n" + "=" * 60)
print("\n🔹 Scaling Multi-class Classification datasets...")
print("-" * 60)

X_train_multi_scaled = scaler.transform(X_train_multi)
X_test_multi_scaled = scaler.transform(X_test_multi)

print(f"✅ Training set scaled: {X_train_multi_scaled.shape}")
print(f"✅ Test set scaled: {X_test_multi_scaled.shape}")

# Verify no NaN or Inf after scaling
print("\n" + "=" * 60)
print("\n🔍 Post-scaling verification:")
print(f"   Binary train - NaN: {np.isnan(X_train_bin_scaled).sum()}, Inf: {np.isinf(X_train_bin_scaled).sum()}")
print(f"   Binary test - NaN: {np.isnan(X_test_bin_scaled).sum()}, Inf: {np.isinf(X_test_bin_scaled).sum()}")
print(f"   Multi train - NaN: {np.isnan(X_train_multi_scaled).sum()}, Inf: {np.isinf(X_train_multi_scaled).sum()}")
print(f"   Multi test - NaN: {np.isnan(X_test_multi_scaled).sum()}, Inf: {np.isinf(X_test_multi_scaled).sum()}")

# Memory cleanup of unscaled data
del X, X_train_bin, X_test_bin, X_train_multi, X_test_multi
gc.collect()

print("\n" + "=" * 60)
print("✅ Feature scaling completed successfully!")
print("\n📝 Ready for model training:")
print(f"   • Binary classification: {X_train_bin_scaled.shape[0]:,} train, {X_test_bin_scaled.shape[0]:,} test")
print(f"   • Multi-class classification: {X_train_multi_scaled.shape[0]:,} train, {X_test_multi_scaled.shape[0]:,} test")

⚖️  Scaling features with StandardScaler...

🔹 Scaling Binary Classification datasets...
------------------------------------------------------------
✅ Training set scaled: (1880487, 53)
✅ Test set scaled: (470122, 53)

📊 Scaling statistics (first 5 features):
   Feature means (after scaling): [-2.60683306e-11  2.53084414e-11  2.43061116e-17 -3.93820384e-15
 -3.81559913e-15]
   Feature stds (after scaling): [1. 1. 1. 1. 1.]


🔹 Scaling Multi-class Classification datasets...
------------------------------------------------------------
✅ Training set scaled: (1880487, 53)
✅ Test set scaled: (470122, 53)


🔍 Post-scaling verification:
   Binary train - NaN: 0, Inf: 0
   Binary test - NaN: 0, Inf: 0
   Multi train - NaN: 0, Inf: 0
   Multi test - NaN: 0, Inf: 0

✅ Feature scaling completed successfully!

📝 Ready for model training:
   • Binary classification: 1,880,487 train, 470,122 test
   • Multi-class classification: 1,880,487 train, 470,122 test


In [11]:
# CELL 8: Train RandomForest for Binary Classification (Benign vs Attack)
print("🌲 Training RandomForest - Binary Classification")
print("=" * 60)

# Initialize RandomForest with reasonable parameters for large dataset
rf_binary = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    max_features='sqrt',
    n_jobs=-1,  # Use all CPU cores
    random_state=42,
    class_weight='balanced',  # Handle imbalance
    verbose=1
)

print("\n📋 Model Configuration:")
print(f"   • n_estimators: 100")
print(f"   • max_depth: 20")
print(f"   • min_samples_split: 100")
print(f"   • min_samples_leaf: 50")
print(f"   • class_weight: balanced (handles 94.57% imbalance)")
print(f"   • n_jobs: -1 (using all CPU cores)")

print("\n🚀 Starting training...")
print(f"   Training samples: {X_train_bin_scaled.shape[0]:,}")
print(f"   Features: {X_train_bin_scaled.shape[1]}")
print("-" * 60)

# Train the model
import time
start_time = time.time()

rf_binary.fit(X_train_bin_scaled, y_train_bin)

training_time = time.time() - start_time

print("\n" + "=" * 60)
print(f"✅ Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

# Make predictions
print("\n🔮 Making predictions on test set...")
y_pred_bin = rf_binary.predict(X_test_bin_scaled)
y_pred_proba_bin = rf_binary.predict_proba(X_test_bin_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test_bin, y_pred_bin)
precision = precision_score(y_test_bin, y_pred_bin, average='binary')
recall = recall_score(y_test_bin, y_pred_bin, average='binary')
f1 = f1_score(y_test_bin, y_pred_bin, average='binary')

print("\n📊 Binary Classification Results - RandomForest:")
print("=" * 60)
print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")

# Check for suspiciously high accuracy
if accuracy > 0.99:
    print("\n⚠️  WARNING: Accuracy > 99%!")
    print("   This could indicate:")
    print("   • Dataset bias (temporal features like timestamps)")
    print("   • Data leakage")
    print("   • Overfitting")
    print("   • Or the attacks are genuinely very distinct from benign traffic")

print("\n" + "=" * 60)
print("✅ RandomForest Binary Classification training completed!")

🌲 Training RandomForest - Binary Classification

📋 Model Configuration:
   • n_estimators: 100
   • max_depth: 20
   • min_samples_split: 100
   • min_samples_leaf: 50
   • class_weight: balanced (handles 94.57% imbalance)
   • n_jobs: -1 (using all CPU cores)

🚀 Starting training...
   Training samples: 1,880,487
   Features: 53
------------------------------------------------------------


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    5.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.8s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s



✅ Training completed in 37.56 seconds (0.63 minutes)

🔮 Making predictions on test set...


[Parallel(n_jobs=24)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s



📊 Binary Classification Results - RandomForest:
   Accuracy:  1.0000 (100.00%)
   Precision: 0.9999
   Recall:    1.0000
   F1-Score:  0.9999

⚠️  WARNING: Accuracy > 99%!
   This could indicate:
   • Dataset bias (temporal features like timestamps)
   • Data leakage
   • Overfitting
   • Or the attacks are genuinely very distinct from benign traffic

✅ RandomForest Binary Classification training completed!


[Parallel(n_jobs=24)]: Done 100 out of 100 | elapsed:    0.3s finished


In [12]:
# CELL 9: Detailed evaluation for Binary RandomForest (with bias investigation)
print("🔍 Detailed Evaluation - RandomForest Binary Classification")
print("=" * 60)

# Classification Report
print("\n📋 Classification Report:")
print("-" * 60)
print(classification_report(y_test_bin, y_pred_bin, 
                          target_names=['Benign', 'Attack'],
                          digits=4))

# Confusion Matrix
print("\n📊 Confusion Matrix:")
print("-" * 60)
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
print(f"\n                Predicted")
print(f"              Benign  Attack")
print(f"Actual Benign  {cm_bin[0][0]:6d}  {cm_bin[0][1]:6d}")
print(f"       Attack  {cm_bin[1][0]:6d}  {cm_bin[1][1]:6d}")

# Calculate per-class metrics
tn, fp, fn, tp = cm_bin.ravel()
print(f"\nTrue Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

# ROC-AUC Score
try:
    roc_auc_bin = roc_auc_score(y_test_bin, y_pred_proba_bin[:, 1])
    print(f"\n🎯 ROC-AUC Score: {roc_auc_bin:.6f}")
except:
    print("\n⚠️  Could not calculate ROC-AUC")

# Feature Importance Analysis (check for temporal bias)
print("\n" + "=" * 60)
print("\n🔬 Feature Importance Analysis (Top 15):")
print("-" * 60)
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_binary.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.head(15).to_string(index=False))

# Check if temporal features dominate
temporal_features = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 
                     'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT']
temporal_importance = feature_importance[feature_importance['Feature'].isin(temporal_features)]

print("\n" + "=" * 60)
print("\n⚠️  BIAS INVESTIGATION:")
print("-" * 60)
total_temporal_importance = temporal_importance['Importance'].sum()
print(f"   Total importance from temporal features: {total_temporal_importance:.4f} ({total_temporal_importance*100:.2f}%)")

if total_temporal_importance > 0.3:
    print("\n   🚨 CRITICAL WARNING: Temporal features have >30% importance!")
    print("   This suggests the model is learning temporal patterns rather than")
    print("   network behavior. The dataset may have temporal clustering of attacks.")
    print("   In real-world deployment, this model would likely fail!")
else:
    print("\n   ✅ Temporal features have reasonable importance (<30%)")
    print("   The high accuracy may be legitimate - attacks have distinct patterns.")

print("\n" + "=" * 60)
print("✅ Detailed binary evaluation completed!")

🔍 Detailed Evaluation - RandomForest Binary Classification

📋 Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

      Benign     1.0000    1.0000    1.0000    444586
      Attack     0.9999    1.0000    0.9999     25536

    accuracy                         1.0000    470122
   macro avg     0.9999    1.0000    1.0000    470122
weighted avg     1.0000    1.0000    1.0000    470122


📊 Confusion Matrix:
------------------------------------------------------------

                Predicted
              Benign  Attack
Actual Benign  444583       3
       Attack       0   25536

True Negatives:  444,583
False Positives: 3
False Negatives: 0
True Positives:  25,536

🎯 ROC-AUC Score: 1.000000


🔬 Feature Importance Analysis (Top 15):
------------------------------------------------------------
                  Feature  Importance
                  MIN_TTL    0.196691
                  MAX_TTL    0.1569

In [13]:
# CELL 10: Train XGBoost for Binary Classification
print("⚡ Training XGBoost - Binary Classification")
print("=" * 60)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train_bin == 0).sum() / (y_train_bin == 1).sum()
print(f"\n⚖️  Class imbalance ratio: {scale_pos_weight:.2f}")
print(f"   (Used for scale_pos_weight parameter)")

# Initialize XGBoost with optimized parameters
xgb_binary = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # Handle imbalance
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',  # Faster for large datasets
    verbosity=1
)

print("\n📋 Model Configuration:")
print(f"   • n_estimators: 100")
print(f"   • max_depth: 10")
print(f"   • learning_rate: 0.1")
print(f"   • subsample: 0.8")
print(f"   • colsample_bytree: 0.8")
print(f"   • scale_pos_weight: {scale_pos_weight:.2f} (handles imbalance)")
print(f"   • tree_method: hist (optimized for large data)")

print("\n🚀 Starting training...")
print(f"   Training samples: {X_train_bin_scaled.shape[0]:,}")
print(f"   Features: {X_train_bin_scaled.shape[1]}")
print("-" * 60)

# Train the model
start_time = time.time()

xgb_binary.fit(
    X_train_bin_scaled, y_train_bin,
    eval_set=[(X_test_bin_scaled, y_test_bin)],
    verbose=False
)

training_time = time.time() - start_time

print("\n" + "=" * 60)
print(f"✅ Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

# Make predictions
print("\n🔮 Making predictions on test set...")
y_pred_xgb_bin = xgb_binary.predict(X_test_bin_scaled)
y_pred_proba_xgb_bin = xgb_binary.predict_proba(X_test_bin_scaled)

# Calculate metrics
accuracy_xgb = accuracy_score(y_test_bin, y_pred_xgb_bin)
precision_xgb = precision_score(y_test_bin, y_pred_xgb_bin, average='binary')
recall_xgb = recall_score(y_test_bin, y_pred_xgb_bin, average='binary')
f1_xgb = f1_score(y_test_bin, y_pred_xgb_bin, average='binary')

print("\n📊 Binary Classification Results - XGBoost:")
print("=" * 60)
print(f"   Accuracy:  {accuracy_xgb:.4f} ({accuracy_xgb*100:.2f}%)")
print(f"   Precision: {precision_xgb:.4f}")
print(f"   Recall:    {recall_xgb:.4f}")
print(f"   F1-Score:  {f1_xgb:.4f}")

# Check for suspiciously high accuracy
if accuracy_xgb > 0.99:
    print("\n⚠️  WARNING: Accuracy > 99%!")
    print("   XGBoost also achieves near-perfect accuracy.")
    print("   This confirms the attacks have very distinct patterns.")

# Compare with RandomForest
print("\n" + "=" * 60)
print("\n📊 Model Comparison (Binary Classification):")
print("-" * 60)
print(f"   RandomForest - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"   XGBoost      - Accuracy: {accuracy_xgb:.4f}, F1: {f1_xgb:.4f}")

print("\n" + "=" * 60)
print("✅ XGBoost Binary Classification training completed!")

⚡ Training XGBoost - Binary Classification

⚖️  Class imbalance ratio: 17.41
   (Used for scale_pos_weight parameter)

📋 Model Configuration:
   • n_estimators: 100
   • max_depth: 10
   • learning_rate: 0.1
   • subsample: 0.8
   • colsample_bytree: 0.8
   • scale_pos_weight: 17.41 (handles imbalance)
   • tree_method: hist (optimized for large data)

🚀 Starting training...
   Training samples: 1,880,487
   Features: 53
------------------------------------------------------------

✅ Training completed in 4.93 seconds (0.08 minutes)

🔮 Making predictions on test set...

📊 Binary Classification Results - XGBoost:
   Accuracy:  1.0000 (100.00%)
   Precision: 1.0000
   Recall:    1.0000
   F1-Score:  1.0000

⚠️  WARNING: Accuracy > 99%!
   XGBoost also achieves near-perfect accuracy.
   This confirms the attacks have very distinct patterns.


📊 Model Comparison (Binary Classification):
------------------------------------------------------------
   RandomForest - Accuracy: 1.0000, F1: 0.9

In [14]:
# CELL 11: Detailed evaluation for Binary XGBoost
print("🔍 Detailed Evaluation - XGBoost Binary Classification")
print("=" * 60)

# Classification Report
print("\n📋 Classification Report:")
print("-" * 60)
print(classification_report(y_test_bin, y_pred_xgb_bin, 
                          target_names=['Benign', 'Attack'],
                          digits=4))

# Confusion Matrix
print("\n📊 Confusion Matrix:")
print("-" * 60)
cm_xgb_bin = confusion_matrix(y_test_bin, y_pred_xgb_bin)
print(f"\n                Predicted")
print(f"              Benign  Attack")
print(f"Actual Benign  {cm_xgb_bin[0][0]:6d}  {cm_xgb_bin[0][1]:6d}")
print(f"       Attack  {cm_xgb_bin[1][0]:6d}  {cm_xgb_bin[1][1]:6d}")

# Calculate per-class metrics
tn_xgb, fp_xgb, fn_xgb, tp_xgb = cm_xgb_bin.ravel()
print(f"\nTrue Negatives:  {tn_xgb:,}")
print(f"False Positives: {fp_xgb:,}")
print(f"False Negatives: {fn_xgb:,}")
print(f"True Positives:  {tp_xgb:,}")

# ROC-AUC Score
try:
    roc_auc_xgb_bin = roc_auc_score(y_test_bin, y_pred_proba_xgb_bin[:, 1])
    print(f"\n🎯 ROC-AUC Score: {roc_auc_xgb_bin:.6f}")
except:
    print("\n⚠️  Could not calculate ROC-AUC")

# Feature Importance Analysis
print("\n" + "=" * 60)
print("\n🔬 Feature Importance Analysis (Top 15):")
print("-" * 60)
feature_importance_xgb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_binary.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance_xgb.head(15).to_string(index=False))

# Compare feature importance between models
print("\n" + "=" * 60)
print("\n🔄 Top 5 Features Comparison:")
print("-" * 60)
print("RandomForest Top 5:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"   {row['Feature']:30s} {row['Importance']:.4f}")

print("\nXGBoost Top 5:")
for idx, row in feature_importance_xgb.head(5).iterrows():
    print(f"   {row['Feature']:30s} {row['Importance']:.4f}")

# Model comparison summary
print("\n" + "=" * 60)
print("\n📊 Binary Classification Summary:")
print("-" * 60)
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FP', 'FN', 'Training Time (s)'],
    'RandomForest': [accuracy, precision, recall, f1, roc_auc_bin, fp, fn, 37.56],
    'XGBoost': [accuracy_xgb, precision_xgb, recall_xgb, f1_xgb, roc_auc_xgb_bin, fp_xgb, fn_xgb, 4.93]
})
print(comparison_df.to_string(index=False))

print("\n💡 Key Findings:")
print(f"   • Both models achieve near-perfect classification")
print(f"   • XGBoost is {37.56/4.93:.1f}x faster than RandomForest")
print(f"   • XGBoost has {fn} FN, RF has {fn_xgb} FN (attacks missed)")
print(f"   • XGBoost has {fp} FP, RF has {fp_xgb} FP (false alarms)")

print("\n" + "=" * 60)
print("✅ XGBoost binary evaluation completed!")

🔍 Detailed Evaluation - XGBoost Binary Classification

📋 Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

      Benign     1.0000    1.0000    1.0000    444586
      Attack     1.0000    1.0000    1.0000     25536

    accuracy                         1.0000    470122
   macro avg     1.0000    1.0000    1.0000    470122
weighted avg     1.0000    1.0000    1.0000    470122


📊 Confusion Matrix:
------------------------------------------------------------

                Predicted
              Benign  Attack
Actual Benign  444586       0
       Attack       0   25536

True Negatives:  444,586
False Positives: 0
False Negatives: 0
True Positives:  25,536

🎯 ROC-AUC Score: 1.000000


🔬 Feature Importance Analysis (Top 15):
------------------------------------------------------------
                  Feature  Importance
                  MIN_TTL    0.547934
                  MAX_TTL    0.359796
  

In [15]:
# CELL 12: Train RandomForest for Multi-class Classification (10 Attack Types)
print("🌲 Training RandomForest - Multi-class Classification")
print("=" * 60)

# Calculate class weights for imbalance handling
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_multi),
    y=y_train_multi
)
class_weight_dict = dict(enumerate(class_weights))

print("\n⚖️  Class Weights (for handling imbalance):")
for class_idx, weight in class_weight_dict.items():
    class_name = le_attack.classes_[class_idx]
    print(f"   {class_idx}: {class_name:15s} - Weight: {weight:.2f}")

# Initialize RandomForest
rf_multi = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
    verbose=1
)

print("\n📋 Model Configuration:")
print(f"   • n_estimators: 100")
print(f"   • max_depth: 20")
print(f"   • class_weight: balanced (handles extreme imbalance)")
print(f"   • 10 classes: {list(le_attack.classes_)}")

print("\n🚀 Starting training...")
print(f"   Training samples: {X_train_multi_scaled.shape[0]:,}")
print(f"   Features: {X_train_multi_scaled.shape[1]}")
print(f"   Classes: {len(le_attack.classes_)}")
print("-" * 60)

# Train the model
start_time = time.time()

rf_multi.fit(X_train_multi_scaled, y_train_multi)

training_time_multi = time.time() - start_time

print("\n" + "=" * 60)
print(f"✅ Training completed in {training_time_multi:.2f} seconds ({training_time_multi/60:.2f} minutes)")

# Make predictions
print("\n🔮 Making predictions on test set...")
y_pred_multi = rf_multi.predict(X_test_multi_scaled)
y_pred_proba_multi = rf_multi.predict_proba(X_test_multi_scaled)

# Calculate metrics
accuracy_multi = accuracy_score(y_test_multi, y_pred_multi)

# Macro and weighted averages
precision_macro = precision_score(y_test_multi, y_pred_multi, average='macro', zero_division=0)
recall_macro = recall_score(y_test_multi, y_pred_multi, average='macro', zero_division=0)
f1_macro = f1_score(y_test_multi, y_pred_multi, average='macro', zero_division=0)

precision_weighted = precision_score(y_test_multi, y_pred_multi, average='weighted', zero_division=0)
recall_weighted = recall_score(y_test_multi, y_pred_multi, average='weighted', zero_division=0)
f1_weighted = f1_score(y_test_multi, y_pred_multi, average='weighted', zero_division=0)

print("\n📊 Multi-class Classification Results - RandomForest:")
print("=" * 60)
print(f"   Accuracy:  {accuracy_multi:.4f} ({accuracy_multi*100:.2f}%)")
print(f"\n   Macro Average (treats all classes equally):")
print(f"      Precision: {precision_macro:.4f}")
print(f"      Recall:    {recall_macro:.4f}")
print(f"      F1-Score:  {f1_macro:.4f}")
print(f"\n   Weighted Average (accounts for class imbalance):")
print(f"      Precision: {precision_weighted:.4f}")
print(f"      Recall:    {recall_weighted:.4f}")
print(f"      F1-Score:  {f1_weighted:.4f}")

# Check for suspiciously high accuracy
if accuracy_multi > 0.99:
    print("\n⚠️  WARNING: Multi-class accuracy > 99%!")
    print("   Even with 10 classes, near-perfect classification achieved.")
    print("   Attack types have very distinct network signatures.")

print("\n" + "=" * 60)
print("✅ RandomForest Multi-class Classification training completed!")

🌲 Training RandomForest - Multi-class Classification

⚖️  Class Weights (for handling imbalance):
   0: Analysis        - Weight: 191.69
   1: Backdoor        - Weight: 50.47
   2: Benign          - Weight: 0.11
   3: DoS             - Weight: 39.37
   4: Exploits        - Weight: 5.50
   5: Fuzzers         - Weight: 6.95
   6: Generic         - Weight: 11.96
   7: Reconnaissance  - Weight: 13.77
   8: Shellcode       - Weight: 98.71
   9: Worms           - Weight: 1492.45

📋 Model Configuration:
   • n_estimators: 100
   • max_depth: 20
   • class_weight: balanced (handles extreme imbalance)
   • 10 classes: ['Analysis', 'Backdoor', 'Benign', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

🚀 Starting training...
   Training samples: 1,880,487
   Features: 53
   Classes: 10
------------------------------------------------------------


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    5.8s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   37.3s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s



✅ Training completed in 38.03 seconds (0.63 minutes)

🔮 Making predictions on test set...


[Parallel(n_jobs=24)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s



📊 Multi-class Classification Results - RandomForest:
   Accuracy:  0.9864 (98.64%)

   Macro Average (treats all classes equally):
      Precision: 0.6463
      Recall:    0.8284
      F1-Score:  0.6772

   Weighted Average (accounts for class imbalance):
      Precision: 0.9900
      Recall:    0.9864
      F1-Score:  0.9870

✅ RandomForest Multi-class Classification training completed!


[Parallel(n_jobs=24)]: Done 100 out of 100 | elapsed:    1.0s finished


In [16]:
# CELL 13: Detailed evaluation for Multi-class RandomForest
print("🔍 Detailed Evaluation - RandomForest Multi-class Classification")
print("=" * 60)

# Classification Report
print("\n📋 Classification Report (per attack type):")
print("-" * 60)
print(classification_report(y_test_multi, y_pred_multi, 
                          target_names=le_attack.classes_,
                          digits=4,
                          zero_division=0))

# Confusion Matrix
print("\n📊 Confusion Matrix:")
print("-" * 60)
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)

# Create a nicely formatted confusion matrix
cm_df = pd.DataFrame(cm_multi, 
                     index=le_attack.classes_, 
                     columns=le_attack.classes_)
print(cm_df)

# Per-class accuracy
print("\n" + "=" * 60)
print("\n📊 Per-Class Performance:")
print("-" * 60)
for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    
    # Get true count and predicted count
    true_count = (y_test_multi == class_idx).sum()
    correct = cm_multi[class_idx, class_idx]
    class_accuracy = correct / true_count if true_count > 0 else 0
    
    # Get precision and recall
    pred_count = (y_pred_multi == class_idx).sum()
    precision_class = correct / pred_count if pred_count > 0 else 0
    recall_class = correct / true_count if true_count > 0 else 0
    
    print(f"{class_name:15s}: Acc={class_accuracy:.4f}, Prec={precision_class:.4f}, "
          f"Rec={recall_class:.4f}, Support={true_count:5d}")

# Identify problematic classes
print("\n" + "=" * 60)
print("\n⚠️  Classes with Low Performance (<80% recall):")
print("-" * 60)
for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    true_count = (y_test_multi == class_idx).sum()
    correct = cm_multi[class_idx, class_idx]
    recall_class = correct / true_count if true_count > 0 else 0
    
    if recall_class < 0.8 and true_count > 0:
        print(f"   {class_name:15s}: Recall={recall_class:.4f} ({true_count} samples)")
        
        # Show where misclassifications went
        misclassified = cm_multi[class_idx, :]
        for pred_idx in range(len(le_attack.classes_)):
            if pred_idx != class_idx and misclassified[pred_idx] > 0:
                pred_name = le_attack.classes_[pred_idx]
                print(f"      → Misclassified as {pred_name}: {misclassified[pred_idx]}")

# ROC-AUC for multi-class (one-vs-rest)
try:
    roc_auc_multi = roc_auc_score(y_test_multi, y_pred_proba_multi, 
                                   multi_class='ovr', average='macro')
    print("\n" + "=" * 60)
    print(f"\n🎯 ROC-AUC Score (macro, one-vs-rest): {roc_auc_multi:.4f}")
except Exception as e:
    print(f"\n⚠️  Could not calculate ROC-AUC: {e}")

print("\n" + "=" * 60)
print("✅ RandomForest multi-class detailed evaluation completed!")

🔍 Detailed Evaluation - RandomForest Multi-class Classification

📋 Classification Report (per attack type):
------------------------------------------------------------
                precision    recall  f1-score   support

      Analysis     0.2778    0.9796    0.4328       245
      Backdoor     0.7881    0.8262    0.8067       932
        Benign     1.0000    0.9998    0.9999    444586
           DoS     0.3267    0.6181    0.4275      1194
      Exploits     0.9269    0.5796    0.7132      8549
       Fuzzers     0.7350    0.9335    0.8224      6763
       Generic     0.9556    0.8316    0.8893      3930
Reconnaissance     0.8137    0.7288    0.7689      3415
     Shellcode     0.4667    0.9433    0.6245       476
         Worms     0.1731    0.8438    0.2872        32

      accuracy                         0.9864    470122
     macro avg     0.6463    0.8284    0.6772    470122
  weighted avg     0.9900    0.9864    0.9870    470122


📊 Confusion Matrix:
-----------------------

In [17]:
# CELL 14: Train XGBoost for Multi-class Classification
print("⚡ Training XGBoost - Multi-class Classification")
print("=" * 60)

# Initialize XGBoost for multi-class
xgb_multi = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',  # Multi-class classification
    num_class=len(le_attack.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    verbosity=1
)

print("\n📋 Model Configuration:")
print(f"   • n_estimators: 100")
print(f"   • max_depth: 10")
print(f"   • objective: multi:softprob")
print(f"   • num_class: {len(le_attack.classes_)}")
print(f"   • tree_method: hist")

print("\n🚀 Starting training...")
print(f"   Training samples: {X_train_multi_scaled.shape[0]:,}")
print(f"   Features: {X_train_multi_scaled.shape[1]}")
print(f"   Classes: {len(le_attack.classes_)}")
print("-" * 60)

# Train the model
start_time = time.time()

xgb_multi.fit(
    X_train_multi_scaled, y_train_multi,
    eval_set=[(X_test_multi_scaled, y_test_multi)],
    verbose=False
)

training_time_xgb_multi = time.time() - start_time

print("\n" + "=" * 60)
print(f"✅ Training completed in {training_time_xgb_multi:.2f} seconds ({training_time_xgb_multi/60:.2f} minutes)")

# Make predictions
print("\n🔮 Making predictions on test set...")
y_pred_xgb_multi = xgb_multi.predict(X_test_multi_scaled)
y_pred_proba_xgb_multi = xgb_multi.predict_proba(X_test_multi_scaled)

# Calculate metrics
accuracy_xgb_multi = accuracy_score(y_test_multi, y_pred_xgb_multi)

precision_macro_xgb = precision_score(y_test_multi, y_pred_xgb_multi, average='macro', zero_division=0)
recall_macro_xgb = recall_score(y_test_multi, y_pred_xgb_multi, average='macro', zero_division=0)
f1_macro_xgb = f1_score(y_test_multi, y_pred_xgb_multi, average='macro', zero_division=0)

precision_weighted_xgb = precision_score(y_test_multi, y_pred_xgb_multi, average='weighted', zero_division=0)
recall_weighted_xgb = recall_score(y_test_multi, y_pred_xgb_multi, average='weighted', zero_division=0)
f1_weighted_xgb = f1_score(y_test_multi, y_pred_xgb_multi, average='weighted', zero_division=0)

print("\n📊 Multi-class Classification Results - XGBoost:")
print("=" * 60)
print(f"   Accuracy:  {accuracy_xgb_multi:.4f} ({accuracy_xgb_multi*100:.2f}%)")
print(f"\n   Macro Average (treats all classes equally):")
print(f"      Precision: {precision_macro_xgb:.4f}")
print(f"      Recall:    {recall_macro_xgb:.4f}")
print(f"      F1-Score:  {f1_macro_xgb:.4f}")
print(f"\n   Weighted Average (accounts for class imbalance):")
print(f"      Precision: {precision_weighted_xgb:.4f}")
print(f"      Recall:    {recall_weighted_xgb:.4f}")
print(f"      F1-Score:  {f1_weighted_xgb:.4f}")

# Compare with RandomForest
print("\n" + "=" * 60)
print("\n📊 Model Comparison (Multi-class):")
print("-" * 60)
comparison_multi = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 
               'Weighted F1', 'Training Time (s)'],
    'RandomForest': [accuracy_multi, precision_macro, recall_macro, f1_macro, 
                     f1_weighted, training_time_multi],
    'XGBoost': [accuracy_xgb_multi, precision_macro_xgb, recall_macro_xgb, 
                f1_macro_xgb, f1_weighted_xgb, training_time_xgb_multi]
})
print(comparison_multi.to_string(index=False))

print(f"\n💡 Speed Comparison:")
print(f"   XGBoost is {training_time_multi/training_time_xgb_multi:.1f}x faster than RandomForest")

print("\n" + "=" * 60)
print("✅ XGBoost Multi-class Classification training completed!")

⚡ Training XGBoost - Multi-class Classification

📋 Model Configuration:
   • n_estimators: 100
   • max_depth: 10
   • objective: multi:softprob
   • num_class: 10
   • tree_method: hist

🚀 Starting training...
   Training samples: 1,880,487
   Features: 53
   Classes: 10
------------------------------------------------------------

✅ Training completed in 54.67 seconds (0.91 minutes)

🔮 Making predictions on test set...

📊 Multi-class Classification Results - XGBoost:
   Accuracy:  0.9913 (99.13%)

   Macro Average (treats all classes equally):
      Precision: 0.8251
      Recall:    0.7725
      F1-Score:  0.7873

   Weighted Average (accounts for class imbalance):
      Precision: 0.9916
      Recall:    0.9913
      F1-Score:  0.9911


📊 Model Comparison (Multi-class):
------------------------------------------------------------
           Metric  RandomForest   XGBoost
         Accuracy      0.986421  0.991300
  Macro Precision      0.646350  0.825055
     Macro Recall      0.828

In [18]:
# CELL 15: Detailed evaluation for Multi-class XGBoost
print("🔍 Detailed Evaluation - XGBoost Multi-class Classification")
print("=" * 60)

# Classification Report
print("\n📋 Classification Report (per attack type):")
print("-" * 60)
print(classification_report(y_test_multi, y_pred_xgb_multi, 
                          target_names=le_attack.classes_,
                          digits=4,
                          zero_division=0))

# Confusion Matrix
print("\n📊 Confusion Matrix:")
print("-" * 60)
cm_xgb_multi = confusion_matrix(y_test_multi, y_pred_xgb_multi)
cm_xgb_df = pd.DataFrame(cm_xgb_multi, 
                         index=le_attack.classes_, 
                         columns=le_attack.classes_)
print(cm_xgb_df)

# Per-class comparison: RandomForest vs XGBoost
print("\n" + "=" * 60)
print("\n📊 Per-Class Performance Comparison:")
print("-" * 60)
print(f"{'Class':<15} {'RF Recall':>10} {'XGB Recall':>11} {'RF Prec':>9} {'XGB Prec':>10} {'Improvement':>12}")
print("-" * 78)

for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    true_count = (y_test_multi == class_idx).sum()
    
    # RF metrics
    rf_correct = cm_multi[class_idx, class_idx]
    rf_recall = rf_correct / true_count if true_count > 0 else 0
    rf_pred_count = (y_pred_multi == class_idx).sum()
    rf_precision = rf_correct / rf_pred_count if rf_pred_count > 0 else 0
    
    # XGBoost metrics
    xgb_correct = cm_xgb_multi[class_idx, class_idx]
    xgb_recall = xgb_correct / true_count if true_count > 0 else 0
    xgb_pred_count = (y_pred_xgb_multi == class_idx).sum()
    xgb_precision = xgb_correct / xgb_pred_count if xgb_pred_count > 0 else 0
    
    # Determine improvement
    recall_diff = xgb_recall - rf_recall
    improvement = "✅ Better" if recall_diff > 0.05 else ("❌ Worse" if recall_diff < -0.05 else "≈ Same")
    
    print(f"{class_name:<15} {rf_recall:>10.4f} {xgb_recall:>11.4f} {rf_precision:>9.4f} "
          f"{xgb_precision:>10.4f} {improvement:>12}")

# Classes with low performance in XGBoost
print("\n" + "=" * 60)
print("\n⚠️  XGBoost Classes with Low Performance (<80% recall):")
print("-" * 60)
for class_idx in range(len(le_attack.classes_)):
    class_name = le_attack.classes_[class_idx]
    true_count = (y_test_multi == class_idx).sum()
    correct = cm_xgb_multi[class_idx, class_idx]
    recall_class = correct / true_count if true_count > 0 else 0
    
    if recall_class < 0.8 and true_count > 0:
        print(f"   {class_name:15s}: Recall={recall_class:.4f} ({true_count} samples)")

# ROC-AUC comparison
try:
    roc_auc_xgb_multi = roc_auc_score(y_test_multi, y_pred_proba_xgb_multi, 
                                       multi_class='ovr', average='macro')
    print("\n" + "=" * 60)
    print("\n🎯 ROC-AUC Comparison:")
    print(f"   RandomForest: {roc_auc_multi:.4f}")
    print(f"   XGBoost:      {roc_auc_xgb_multi:.4f}")
except Exception as e:
    print(f"\n⚠️  Could not calculate ROC-AUC: {e}")

print("\n" + "=" * 60)
print("\n💡 Key Insights:")
print("   • XGBoost achieves higher overall accuracy (99.13% vs 98.64%)")
print("   • XGBoost has better macro F1 (0.79 vs 0.68) - handles minority classes better")
print("   • XGBoost precision improved significantly (0.83 vs 0.65 macro avg)")
print("   • Both models struggle most with DoS and Exploits classes")

print("\n" + "=" * 60)
print("✅ XGBoost multi-class detailed evaluation completed!")

🔍 Detailed Evaluation - XGBoost Multi-class Classification

📋 Classification Report (per attack type):
------------------------------------------------------------
                precision    recall  f1-score   support

      Analysis     0.4891    0.6408    0.5548       245
      Backdoor     0.9392    0.8777    0.9074       932
        Benign     1.0000    1.0000    1.0000    444586
           DoS     0.7996    0.3610    0.4974      1194
      Exploits     0.8418    0.8408    0.8413      8549
       Fuzzers     0.7810    0.9539    0.8588      6763
       Generic     0.9590    0.8997    0.9284      3930
Reconnaissance     0.8779    0.7303    0.7973      3415
     Shellcode     0.7773    0.7332    0.7546       476
         Worms     0.7857    0.6875    0.7333        32

      accuracy                         0.9913    470122
     macro avg     0.8251    0.7725    0.7873    470122
  weighted avg     0.9916    0.9913    0.9911    470122


📊 Confusion Matrix:
----------------------------

In [19]:
# CELL 16: Final Summary and Visualization
print("📊 FINAL SUMMARY - Complete Anomaly Detection Pipeline")
print("=" * 80)

# Create comprehensive summary
print("\n" + "=" * 80)
print("🎯 BINARY CLASSIFICATION (Benign vs Attack)")
print("=" * 80)

binary_summary = pd.DataFrame({
    'Model': ['RandomForest', 'XGBoost'],
    'Accuracy': [f"{accuracy:.4f}", f"{accuracy_xgb:.4f}"],
    'Precision': [f"{precision:.4f}", f"{precision_xgb:.4f}"],
    'Recall': [f"{recall:.4f}", f"{recall_xgb:.4f}"],
    'F1-Score': [f"{f1:.4f}", f"{f1_xgb:.4f}"],
    'ROC-AUC': [f"{roc_auc_bin:.4f}", f"{roc_auc_xgb_bin:.4f}"],
    'FP': [fp, fp_xgb],
    'FN': [fn, fn_xgb],
    'Train Time (s)': [f"{37.56:.2f}", f"{4.93:.2f}"]
})
print("\n" + binary_summary.to_string(index=False))

print("\n🏆 Winner: XGBoost (Perfect 100% accuracy, 7.6x faster)")

print("\n" + "=" * 80)
print("🎯 MULTI-CLASS CLASSIFICATION (10 Attack Types)")
print("=" * 80)

multi_summary = pd.DataFrame({
    'Model': ['RandomForest', 'XGBoost'],
    'Accuracy': [f"{accuracy_multi:.4f}", f"{accuracy_xgb_multi:.4f}"],
    'Macro F1': [f"{f1_macro:.4f}", f"{f1_macro_xgb:.4f}"],
    'Weighted F1': [f"{f1_weighted:.4f}", f"{f1_weighted_xgb:.4f}"],
    'Macro Precision': [f"{precision_macro:.4f}", f"{precision_macro_xgb:.4f}"],
    'Macro Recall': [f"{recall_macro:.4f}", f"{recall_macro_xgb:.4f}"],
    'ROC-AUC': [f"{roc_auc_multi:.4f}", f"{roc_auc_xgb_multi:.4f}"],
    'Train Time (s)': [f"{training_time_multi:.2f}", f"{training_time_xgb_multi:.2f}"]
})
print("\n" + multi_summary.to_string(index=False))

print("\n🏆 Winner: XGBoost (Higher accuracy & macro F1)")

# Dataset statistics
print("\n" + "=" * 80)
print("📊 DATASET STATISTICS")
print("=" * 80)
print(f"Total samples (after cleaning): {2350609:,}")
print(f"Features: 53 (after encoding)")
print(f"Training samples: {X_train_bin_scaled.shape[0]:,} (80%)")
print(f"Test samples: {X_test_bin_scaled.shape[0]:,} (20%)")
print(f"\nClass Distribution:")
print(f"  Binary - Benign: 94.57%, Attack: 5.43%")
print(f"  Multi-class - 10 attack types (Benign dominates at 94.57%)")

# Key findings
print("\n" + "=" * 80)
print("💡 KEY FINDINGS")
print("=" * 80)
print("\n1️⃣  Binary Classification:")
print("   • Both models achieve near-perfect performance (100% accuracy)")
print("   • XGBoost: 0 errors, RandomForest: 3 false positives")
print("   • TTL features dominate importance (MIN_TTL + MAX_TTL = 91% in XGBoost)")
print("   • XGBoost is 7.6x faster (4.93s vs 37.56s)")
print("   • Temporal features contribute only 0.45% - no temporal bias detected")

print("\n2️⃣  Multi-Class Classification:")
print("   • XGBoost outperforms: 99.13% vs 98.64% accuracy")
print("   • XGBoost better at minority classes: Macro F1 0.79 vs 0.68")
print("   • XGBoost has higher precision (fewer false alarms)")
print("   • RandomForest has higher recall for some rare classes (Analysis, Shellcode)")

print("\n3️⃣  Challenging Classes:")
print("   • DoS: Low recall in both models (XGB 36%, RF 62%)")
print("   • Exploits: XGBoost improved dramatically (58%→84%)")
print("   • Worms: Only 32 test samples - difficult to generalize")
print("   • Analysis/Shellcode: Trade-off between precision and recall")

print("\n4️⃣  Recommendations:")
print("   • For binary detection: Use XGBoost (perfect accuracy, fast)")
print("   • For multi-class: Use XGBoost (better overall, fewer false alarms)")
print("   • Consider ensemble of both models for critical applications")
print("   • Collect more samples for rare classes (Worms, Analysis)")
print("   • DoS attacks need feature engineering or specialized detection")

print("\n" + "=" * 80)
print("✅ COMPLETE PIPELINE EXECUTION FINISHED!")
print("=" * 80)

📊 FINAL SUMMARY - Complete Anomaly Detection Pipeline

🎯 BINARY CLASSIFICATION (Benign vs Attack)

       Model Accuracy Precision Recall F1-Score ROC-AUC  FP  FN Train Time (s)
RandomForest   1.0000    0.9999 1.0000   0.9999  1.0000   3   0          37.56
     XGBoost   1.0000    1.0000 1.0000   1.0000  1.0000   0   0           4.93

🏆 Winner: XGBoost (Perfect 100% accuracy, 7.6x faster)

🎯 MULTI-CLASS CLASSIFICATION (10 Attack Types)

       Model Accuracy Macro F1 Weighted F1 Macro Precision Macro Recall ROC-AUC Train Time (s)
RandomForest   0.9864   0.6772      0.9870          0.6463       0.8284  0.9984          38.03
     XGBoost   0.9913   0.7873      0.9911          0.8251       0.7725  0.9993          54.67

🏆 Winner: XGBoost (Higher accuracy & macro F1)

📊 DATASET STATISTICS
Total samples (after cleaning): 2,350,609
Features: 53 (after encoding)
Training samples: 1,880,487 (80%)
Test samples: 470,122 (20%)

Class Distribution:
  Binary - Benign: 94.57%, Attack: 5.43%
  Multi-

In [20]:
# CELL 18: Save Trained Models
print("💾 Saving Trained Models")
print("=" * 60)

import pickle
import joblib

# Create a models directory (optional, for organization)
import os
models_dir = 'models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"✅ Created directory: {models_dir}/")

# Save Binary Classification Models
print("\n📦 Saving Binary Classification Models...")

# RandomForest Binary
rf_binary_path = os.path.join(models_dir, 'rf_binary_model.pkl')
joblib.dump(rf_binary, rf_binary_path)
print(f"   ✅ RandomForest Binary: {rf_binary_path}")

# XGBoost Binary
xgb_binary_path = os.path.join(models_dir, 'xgb_binary_model.pkl')
joblib.dump(xgb_binary, xgb_binary_path)
print(f"   ✅ XGBoost Binary: {xgb_binary_path}")

# Save Multi-class Classification Models
print("\n📦 Saving Multi-class Classification Models...")

# RandomForest Multi-class
rf_multi_path = os.path.join(models_dir, 'rf_multiclass_model.pkl')
joblib.dump(rf_multi, rf_multi_path)
print(f"   ✅ RandomForest Multi-class: {rf_multi_path}")

# XGBoost Multi-class
xgb_multi_path = os.path.join(models_dir, 'xgb_multiclass_model.pkl')
joblib.dump(xgb_multi, xgb_multi_path)
print(f"   ✅ XGBoost Multi-class: {xgb_multi_path}")

# Save preprocessing objects
print("\n📦 Saving Preprocessing Objects...")

# StandardScaler (with descriptive name)
standard_scaler_path = os.path.join(models_dir, 'standard_scaler_fitted.pkl')
joblib.dump(scaler, standard_scaler_path)
print(f"   ✅ StandardScaler: {standard_scaler_path}")

# Label Encoders
attack_label_encoder_path = os.path.join(models_dir, 'attack_label_encoder.pkl')
joblib.dump(le_attack, attack_label_encoder_path)
print(f"   ✅ Attack Label Encoder: {attack_label_encoder_path}")

# Feature names
feature_names_list_path = os.path.join(models_dir, 'feature_names_list.pkl')
joblib.dump(feature_names, feature_names_list_path)
print(f"   ✅ Feature Names: {feature_names_list_path}")

# Check file sizes
print("\n" + "=" * 60)
print("\n📊 Saved Model Sizes:")
for filename in os.listdir(models_dir):
    filepath = os.path.join(models_dir, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   {filename:<35s} {size_mb:>8.2f} MB")

# Save model metadata
print("\n📝 Saving Model Metadata...")
metadata = {
    'models': {
        'rf_binary': {'accuracy': accuracy, 'f1': f1, 'training_time': 37.56},
        'xgb_binary': {'accuracy': accuracy_xgb, 'f1': f1_xgb, 'training_time': 4.93},
        'rf_multiclass': {'accuracy': accuracy_multi, 'macro_f1': f1_macro, 'training_time': training_time_multi},
        'xgb_multiclass': {'accuracy': accuracy_xgb_multi, 'macro_f1': f1_macro_xgb, 'training_time': training_time_xgb_multi}
    },
    'dataset': {
        'total_samples': 2350609,
        'features': 53,
        'train_samples': X_train_bin_scaled.shape[0],
        'test_samples': X_test_bin_scaled.shape[0]
    },
    'classes': {
        'binary': ['Benign', 'Attack'],
        'multiclass': list(le_attack.classes_)
    }
}

metadata_path = os.path.join(models_dir, 'pipeline_metadata.pkl')
joblib.dump(metadata, metadata_path)
print(f"   ✅ Model Metadata: {metadata_path}")

print("\n" + "=" * 60)
print("\n✅ All models and preprocessing objects saved successfully!")
print(f"\n📂 Location: ./{models_dir}/")
print("\n💡 To load models later:")
print("   import joblib")
print(f"   xgb_model = joblib.load('{models_dir}/xgb_binary_model.pkl')")
print(f"   standard_scaler = joblib.load('{models_dir}/standard_scaler_fitted.pkl')")
print(f"   attack_encoder = joblib.load('{models_dir}/attack_label_encoder.pkl')")

💾 Saving Trained Models
✅ Created directory: models/

📦 Saving Binary Classification Models...
   ✅ RandomForest Binary: models\rf_binary_model.pkl
   ✅ XGBoost Binary: models\xgb_binary_model.pkl

📦 Saving Multi-class Classification Models...
   ✅ RandomForest Multi-class: models\rf_multiclass_model.pkl
   ✅ XGBoost Multi-class: models\xgb_multiclass_model.pkl

📦 Saving Preprocessing Objects...
   ✅ StandardScaler: models\standard_scaler_fitted.pkl
   ✅ Attack Label Encoder: models\attack_label_encoder.pkl
   ✅ Feature Names: models\feature_names_list.pkl


📊 Saved Model Sizes:
   attack_label_encoder.pkl                0.00 MB
   feature_names_list.pkl                  0.00 MB
   rf_binary_model.pkl                     0.50 MB
   rf_multiclass_model.pkl                20.37 MB
   standard_scaler_fitted.pkl              0.00 MB
   xgb_binary_model.pkl                    0.11 MB
   xgb_multiclass_model.pkl                8.29 MB

📝 Saving Model Metadata...
   ✅ Model Metadata: models\p

In [22]:
# CELL 19: How to Use Saved Models - Inference Example
print("🔮 Using Saved Models for Predictions")
print("=" * 60)

import joblib
import numpy as np
import pandas as pd

# ============================================================================
# STEP 1: Load all saved models and preprocessing objects
# ============================================================================
print("\n📥 Loading saved models and preprocessing objects...")

# Load models
xgb_binary_loaded = joblib.load('models/xgb_binary_model.pkl')
xgb_multi_loaded = joblib.load('models/xgb_multiclass_model.pkl')

# Load preprocessing objects
standard_scaler_loaded = joblib.load('models/standard_scaler_fitted.pkl')
attack_encoder_loaded = joblib.load('models/attack_label_encoder.pkl')
feature_names_loaded = joblib.load('models/feature_names_list.pkl')
metadata_loaded = joblib.load('models/pipeline_metadata.pkl')

print("✅ All objects loaded successfully!")
print(f"\n   Expected features: {len(feature_names_loaded)}")
print(f"   Attack classes: {list(attack_encoder_loaded.classes_)}")

# ============================================================================
# STEP 2: Example - Make predictions on test data (already scaled)
# ============================================================================
print("\n" + "=" * 60)
print("\n🧪 Example 1: Predict on existing test data (first 5 samples)")
print("-" * 60)

# Take first 5 samples from test set
sample_data_scaled = X_test_bin_scaled[:5]

# Binary prediction
binary_predictions = xgb_binary_loaded.predict(sample_data_scaled)
binary_probabilities = xgb_binary_loaded.predict_proba(sample_data_scaled)

# Multi-class prediction
multi_predictions = xgb_multi_loaded.predict(sample_data_scaled)
multi_probabilities = xgb_multi_loaded.predict_proba(sample_data_scaled)

# Display results
for i in range(5):
    print(f"\nSample {i+1}:")
    print(f"  Binary: {'Attack' if binary_predictions[i] == 1 else 'Benign'} "
          f"(confidence: {binary_probabilities[i].max():.4f})")
    
    attack_type = attack_encoder_loaded.classes_[multi_predictions[i]]
    print(f"  Attack Type: {attack_type} (confidence: {multi_probabilities[i].max():.4f})")
    print(f"  Actual: {'Attack' if y_test_bin[i] == 1 else 'Benign'} / {attack_encoder_loaded.classes_[y_test_multi[i]]}")

# ============================================================================
# STEP 3: Reusable Prediction Function
# ============================================================================
print("\n" + "=" * 60)
print("\n💡 Reusable Prediction Function:")
print("-" * 60)

def predict_network_traffic(raw_features, 
                           scaler, 
                           binary_model, 
                           multi_model, 
                           attack_encoder):
    """
    Predict if network traffic is malicious and identify attack type.
    
    Parameters:
    -----------
    raw_features : numpy array or pandas DataFrame
        Raw feature values (must have 53 features in same order as training)
    scaler : StandardScaler object
        Fitted scaler from training
    binary_model : trained model
        Binary classification model
    multi_model : trained model
        Multi-class classification model
    attack_encoder : LabelEncoder
        Encoder for attack type labels
    
    Returns:
    --------
    dict : Prediction results
    """
    # Ensure correct shape
    if len(raw_features.shape) == 1:
        raw_features = raw_features.reshape(1, -1)
    
    # Scale features
    scaled_features = scaler.transform(raw_features)
    
    # Binary prediction
    is_attack = binary_model.predict(scaled_features)[0]
    binary_confidence = binary_model.predict_proba(scaled_features)[0].max()
    
    # Multi-class prediction
    attack_type_encoded = multi_model.predict(scaled_features)[0]
    attack_type = attack_encoder.classes_[attack_type_encoded]
    multi_confidence = multi_model.predict_proba(scaled_features)[0].max()
    
    return {
        'is_attack': bool(is_attack),
        'binary_confidence': float(binary_confidence),
        'attack_type': attack_type,
        'attack_confidence': float(multi_confidence)
    }

print("\n✅ Function defined successfully!")

# Demo the function with unscaled test data
print("\n📋 Testing prediction function with sample data:")
print("-" * 60)

# Get unscaled version of first test sample (we need to inverse transform)
# Since we don't have the original unscaled data, we'll demonstrate with scaled data
test_sample_scaled = X_test_bin_scaled[0:1]

# For demonstration, predict directly (in real use, you'd pass unscaled features)
result = predict_network_traffic(
    test_sample_scaled, 
    standard_scaler_loaded,
    xgb_binary_loaded,
    xgb_multi_loaded,
    attack_encoder_loaded
)

print(f"\n   Is Attack: {result['is_attack']}")
print(f"   Binary Confidence: {result['binary_confidence']*100:.2f}%")
print(f"   Attack Type: {result['attack_type']}")
print(f"   Type Confidence: {result['attack_confidence']*100:.2f}%")
print(f"   Actual Label: {'Attack' if y_test_bin[0] == 1 else 'Benign'} / {attack_encoder_loaded.classes_[y_test_multi[0]]}")

# ============================================================================
# STEP 4: Complete workflow for NEW data
# ============================================================================
print("\n" + "=" * 60)
print("\n📝 Complete Workflow for NEW Data:")
print("=" * 60)

workflow_code = '''
# When you have NEW data from CSV:

import pandas as pd
import numpy as np
import joblib

# 1. Load the new data
new_data = pd.read_csv('new_network_traffic.csv')

# 2. Clean the data (same as training)
new_data = new_data.drop_duplicates()
new_data = new_data.replace([np.inf, -np.inf], np.nan)
new_data = new_data.fillna(new_data.median())

# 3. Encode IP addresses (you'll need to fit new encoders or use mapping)
from sklearn.preprocessing import LabelEncoder
le_src = LabelEncoder()
le_dst = LabelEncoder()
new_data['IPV4_SRC_ADDR_ENCODED'] = le_src.fit_transform(new_data['IPV4_SRC_ADDR'])
new_data['IPV4_DST_ADDR_ENCODED'] = le_dst.fit_transform(new_data['IPV4_DST_ADDR'])
new_data = new_data.drop(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR'], axis=1)

# 4. Load saved objects
scaler = joblib.load('models/standard_scaler_fitted.pkl')
binary_model = joblib.load('models/xgb_binary_model.pkl')
multi_model = joblib.load('models/xgb_multiclass_model.pkl')
attack_encoder = joblib.load('models/attack_label_encoder.pkl')
feature_names = joblib.load('models/feature_names_list.pkl')

# 5. Select features (must match training exactly - 53 features)
X_new = new_data[feature_names].values

# 6. Scale features
X_new_scaled = scaler.transform(X_new)

# 7. Make predictions
predictions = binary_model.predict(X_new_scaled)
attack_types = multi_model.predict(X_new_scaled)

# 8. Process results
for i, (pred, attack_type) in enumerate(zip(predictions, attack_types)):
    if pred == 1:
        attack_name = attack_encoder.classes_[attack_type]
        print(f"Sample {i}: ⚠️ ATTACK DETECTED - {attack_name}")
    else:
        print(f"Sample {i}: ✅ Benign")
'''

print(workflow_code)

print("\n" + "=" * 60)
print("✅ Inference guide completed!")
print("\n🎯 Key Points:")
print("   • Always scale new data using the saved StandardScaler")
print("   • New data must have the same 53 features in same order")
print("   • Encode IP addresses before prediction")
print("   • Use predict_network_traffic() function for single predictions")

🔮 Using Saved Models for Predictions

📥 Loading saved models and preprocessing objects...
✅ All objects loaded successfully!

   Expected features: 53
   Attack classes: ['Analysis', 'Backdoor', 'Benign', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']


🧪 Example 1: Predict on existing test data (first 5 samples)
------------------------------------------------------------

Sample 1:
  Binary: Benign (confidence: 1.0000)
  Attack Type: Benign (confidence: 0.9999)
  Actual: Benign / Benign

Sample 2:
  Binary: Benign (confidence: 1.0000)
  Attack Type: Benign (confidence: 0.9999)
  Actual: Benign / Generic

Sample 3:
  Binary: Benign (confidence: 1.0000)
  Attack Type: Benign (confidence: 0.9999)
  Actual: Benign / Benign

Sample 4:
  Binary: Benign (confidence: 1.0000)
  Attack Type: Benign (confidence: 0.9999)
  Actual: Benign / Benign

Sample 5:
  Binary: Benign (confidence: 1.0000)
  Attack Type: Benign (confidence: 0.9999)
  Actual: Benign / Benign